# Mesh refinement

A beam extracted from a plasma through a round aperture, on a coarse mesh with a refined
box where the field changes fastest: the plasma meniscus and the aperture. The box gives
the beam of a mesh with half the cells everywhere, with half its nodes. The Vlasov
iteration can also place such boxes itself (`AdaptiveRefinementParams`).

A box is a `RefinementParams(level, lower, upper)` in `BeamOpticsConfig.refinement`: its
cells are $2^\text{level}$ times smaller than the grid's. The field is solved on the
composite mesh of the grid and its boxes (patchwork's), and every trajectory steps and
deposits its charge on the finest box it is in. Boxes work on 2D, cylindrical and 3D grids.

In [ ]:
import numpy as np
import plasma_plots as pp

from beamtracer import (
    AdaptiveRefinementParams,
    BeamOptics,
    BeamOpticsConfig,
    BeamSpecies,
    ExtractionPlasma,
    FieldBCs,
    FieldBoundaryCondition,
    GridParams,
    ObjectParams,
    RefinementParams,
)

BLUE, ORANGE, GREY = "C0", "C1", "0.6"
pp.set_backend("plotly")  # interactive figures, embedded in the docs
pp.set_math("latex")  # formulas in the figures typeset as LaTeX

## The extraction

Protons from a plasma at 5 V (Boltzmann electrons at 5 eV, the plasma's edge starting
at $x = 0.6$ mm) are extracted through the 1 mm aperture of the plasma electrode (0 V) by
the extraction electrode at -2 kV, in cylindrical $x$-$r$ geometry. The coarse grid has
0.1 mm cells.

In [ ]:
def plate(x0: float, x1: float, r0: float, voltage: float) -> ObjectParams:
    # an electrode around the axis from the radius r0 outwards, between x0 and x1
    return ObjectParams(
        type="aperture2d",
        axis=0,
        position=0.0,
        vertices=((x0, r0), (x0, 3.1e-3), (x1, 3.1e-3), (x1, r0)),
        bc_type="dirichlet",
        bc_value=voltage,
    )


def grid(cells_per_mm: int) -> GridParams:
    return GridParams(
        num_gridpoints=(8 * cells_per_mm + 1, 3 * cells_per_mm + 1),
        lower_bound=(0.0, 0.0),
        upper_bound=(8e-3, 3e-3),
        coordinates="cylindrical",
    )


neumann = FieldBoundaryCondition.neumann(0.0)
coarse = BeamOpticsConfig(
    grid=grid(10),
    bcs_field=FieldBCs(
        x_lower=FieldBoundaryCondition.dirichlet(5.0),
        x_upper=FieldBoundaryCondition.dirichlet(-2000.0),
        y_lower=neumann,
        y_upper=neumann,
    ),
    objects={
        "plasma electrode": plate(0.5e-3, 1.0e-3, 1.0e-3, 0.0),
        "extraction electrode": plate(3.0e-3, 4.0e-3, 1.2e-3, -2000.0),
    },
    beams=[
        BeamSpecies(
            "H+",
            50.0,
            (0.0, 0.0),
            (0.0, 0.9e-3),
            mass=1.0,
            energy=5.0,
            transverse_temperature=0.5,
            num_particles=100,
        )
    ],  # fmt: skip
    plasma=ExtractionPlasma(5.0, 5.0, 0.6e-3),
    iterations=60,
    tolerance=1e-5,
    relaxation=0.8,
    diagnostic_planes=[7.5e-3],
    save_trajectories=4,
)

## The refined box

One box of level 1 (0.05 mm cells) over the plasma, the meniscus, the aperture and the gap
up to the extraction electrode. Beyond it the beam drifts in a smooth field, where the
coarse cells are enough.

In [ ]:
box = RefinementParams(1, (0.0, 0.0), (4.5e-3, 2.0e-3))
refined = coarse.replace(refinement=[box])
model = BeamOptics(refined).run()
layout = model.state.layout
for patch in layout.patches:
    mesh = patch.mesh
    upper = [o + mesh.h * (n - 1) for o, n in zip(mesh.origin, mesh.shape, strict=True)]
    print(
        f"level {patch.level}: {mesh.shape[0]} x {mesh.shape[1]} nodes, "
        f"{mesh.h * 1e3:.3f} mm cells, x up to {upper[0] * 1e3:.1f} mm, r up to {upper[1] * 1e3:.1f} mm"
    )
print(f"{layout.size} nodes on the composite mesh")

In [ ]:
def cell_lines(x0, x1, r0, r1, h):
    # the lines of the cells of [x0, x1] x [r0, r1], as one path with gaps
    xs, rs = [], []
    for x in np.arange(x0, x1 + h / 2, h):
        xs += [x, x, np.nan]
        rs += [r0, r1, np.nan]
    for r in np.arange(r0, r1 + h / 2, h):
        xs += [x0, x1, np.nan]
        rs += [r, r, np.nan]
    return np.array(xs) * 1e3, np.array(rs) * 1e3


with pp.figure(title="The cells around the aperture") as fig:
    ax = fig[0]
    ax.plot(
        *cell_lines(0.0, 6e-3, 0.0, 3e-3, 1e-4),
        color=GREY,
        lw=0.5,
        label="grid, 0.1 mm",
    )
    ax.plot(
        *cell_lines(0.0, 4.5e-3, 0.0, 2e-3, 5e-5),
        color=BLUE,
        lw=0.5,
        label="box, 0.05 mm",
    )
    for number, electrode in enumerate(refined.objects.values()):
        x, r = np.array(electrode.vertices + electrode.vertices[:1]).T * 1e3
        ax.plot(x, r, color="k", lw=2, label="electrodes" if number == 0 else None)
    ax.set_xlim(0, 6)
    ax.set_ylim(0, 3)
    ax.set_xlabel("$x$ [mm]")
    ax.set_ylabel("$r$ [mm]")
    ax.set_aspect("equal")
    ax.legend()

## Boxes placed by the iteration

With `adaptive_refinement` the iteration places the boxes itself: every few iterations it
tags the cells near the beam across which the field strength, or the beam's charge
density, changes by more than a fraction (`threshold`, 2%) of its largest value, grows
the tags by a buffer of cells and covers them with boxes (Berger--Rigoutsos clustering),
then solves and traces on them. The boxes it chooses here follow the beam from the plasma
through the aperture.

In [ ]:
adaptive = coarse.replace(adaptive_refinement=AdaptiveRefinementParams())
automatic = BeamOptics(adaptive).run()
for patch in automatic.state.layout.patches[1:]:
    mesh = patch.mesh
    upper = [o + mesh.h * (n - 1) for o, n in zip(mesh.origin, mesh.shape, strict=True)]
    print(
        f"level {patch.level}: x {mesh.origin[0] * 1e3:.2f} to {upper[0] * 1e3:.2f} mm, "
        f"r {mesh.origin[1] * 1e3:.2f} to {upper[1] * 1e3:.2f} mm"
    )
print(f"{automatic.state.layout.size} nodes on the composite mesh")
print("boxes per iteration:", [h["boxes"] for h in automatic.convergence_history])

In [ ]:
with pp.figure(title="The cells the iteration refined") as fig:
    ax = fig[0]
    ax.plot(
        *cell_lines(0.0, 8e-3, 0.0, 3e-3, 1e-4),
        color=GREY,
        lw=0.5,
        label="grid, 0.1 mm",
    )
    for number, patch in enumerate(automatic.state.layout.patches[1:]):
        mesh = patch.mesh
        upper = [
            o + mesh.h * (n - 1) for o, n in zip(mesh.origin, mesh.shape, strict=True)
        ]
        ax.plot(
            *cell_lines(mesh.origin[0], upper[0], mesh.origin[1], upper[1], mesh.h),
            color=ORANGE, lw=0.5, label="adaptive box" if number == 0 else None,
        )  # fmt: skip
    for number, electrode in enumerate(refined.objects.values()):
        x, r = np.array(electrode.vertices + electrode.vertices[:1]).T * 1e3
        ax.plot(x, r, color="k", lw=2, label="electrodes" if number == 0 else None)
    for path in automatic.trajectories:
        ax.plot(path[:, 0] * 1e3, path[:, 1] * 1e3, color=BLUE, lw=0.8)
    ax.set_xlim(0, 8)
    ax.set_ylim(0, 3)
    ax.set_xlabel("$x$ [mm]")
    ax.set_ylabel("$r$ [mm]")
    ax.set_aspect("equal")
    ax.legend()

## The potential and the beam

The potential on the grid's nodes (the composite potential restricted to them), the
trajectories and the box.

In [ ]:
ds = model.dataset()
with pp.figure(title="Potential and trajectories") as fig:
    ax = fig[0]
    mesh = ax.pcolormesh(
        ds.x * 1e3, ds.r * 1e3, ds.potential.T, cmap="RdBu_r", shading="auto"
    )
    ax.figure.colorbar(mesh, ax=ax, label=r"$\phi$ [V]")
    for path in model.trajectories:
        ax.plot(path[:, 0] * 1e3, path[:, 1] * 1e3, color="k", lw=0.6)
    (x0, r0), (x1, r1) = np.array(box.lower) * 1e3, np.array(box.upper) * 1e3
    ax.plot([x0, x1, x1, x0, x0], [r0, r0, r1, r1, r0], color=ORANGE, lw=2)
    ax.set_xlabel("$x$ [mm]")
    ax.set_ylabel("$r$ [mm]")

## Against uniform meshes

The rms divergence of the beam at $x = 7.5$ mm on uniform meshes with 0.1, 0.05 and
0.025 mm cells, and on the coarse mesh with the box and with the adaptive boxes. The
finest mesh is the reference.

In [ ]:
def divergence(config: BeamOpticsConfig) -> tuple[int, float]:
    run = BeamOptics(config.replace(save_trajectories=0)).run()
    composite = run.state.composite_potential
    nodes = composite.size if composite is not None else run.potential.size
    return nodes, run.beam_moments(7.5e-3).rms_angle * 1e3


uniform = [divergence(coarse.replace(grid=grid(n))) for n in (10, 20, 40)]
refined_point = divergence(refined)
adaptive_point = divergence(adaptive)
reference = uniform[-1][1]
for label, (nodes, angle) in zip(
    ("0.1 mm", "0.05 mm", "0.025 mm", "0.1 mm with the box", "0.1 mm, adaptive"),
    [*uniform, refined_point, adaptive_point],
    strict=True,
):
    print(f"{label:20s} {nodes:6d} nodes: {angle:.3f} mrad ({angle - reference:+.3f})")

In [ ]:
with pp.figure(title="Divergence against the nodes") as fig:
    ax = fig[0]
    nodes, angles = np.array(uniform).T
    ax.plot(nodes, angles, "o-", color=GREY, label="uniform cells")
    ax.plot(
        *refined_point,
        "s",
        color=BLUE,
        markersize=10,
        label="coarse cells with the box",
    )
    ax.plot(
        *adaptive_point,
        "D",
        color=ORANGE,
        markersize=10,
        label="coarse cells, adaptive",
    )
    ax.axhline(reference, color="k", ls="--", lw=1, label="0.025 mm cells")
    ax.set_xscale("log")
    ax.set_xlabel("nodes")
    ax.set_ylabel(r"rms divergence [mrad]")
    ax.legend()

The coarse mesh overestimates the divergence by 0.4 mrad. The box, placed by hand or
by the iteration, takes it to within about 0.15 mrad of the reference, as close as halving
every cell, with about half the nodes of that mesh. The cells outside the boxes hardly
matter: the beam's divergence is set where it is extracted. (Nearby runs differ by about
0.1 mrad: a few trajectories pass close to the electrodes' edges.)